# 🌐 DeepNT+: Neural Network Tomography with Hybrid Convex Gating, Temporal Message Passing & Calibrated Uncertainty

**Author**: Atikmorshed Sohan  
**Dataset**: [DeepNT-Plus Network Tomography Suite](https://www.kaggle.com/datasets/atikmorshedsohan/deepnt-plus-network-tomography-suite)  
**Code Repository**: [DeepNT_Pluse (GitHub)](https://github.com/Atikmorshedsohan/DeepNT_Pluse)  
**Execution Environment**: Kaggle CPU / GPU (Tesla T4)  

---

### 📌 Reviewer Summary & Purpose of This Notebook
This notebook serves as an interactive **reproducibility and code review hub** for the thesis project **DeepNT+**. It provides:
1. **Full Source Code Inspection**: Direct access to `deepnt_plus` architecture, GNN layers, temporal TGN branch, convex tomography solver, and GeoLite2 pipeline.
2. **Topological & Data Verification**: Live inspection of the newly constructed 10-graph IPv4 and IPv6 topologies, tomography contract checks ($A \cdot \mathbf{1} = \text{hops}$), and GeoLite2 geolocation resolution stats.
3. **Interactive Inference**: A live forward-pass demonstration showing how GNN representations, convex ridge tomography, and gating dynamically fuse.
4. **Authentic 5-Seed Benchmark Results**: Master comparison tables and high-resolution publication charts across all four benchmark suites (IPv4 10-graphs, IPv6 10-graphs, run_03 dense, and run_02 15-graphs) comparing Pure Topological vs GeoLite2 Side-Information tracks.

In [1]:
import os
import sys
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown, HTML

# 1. Locate Dataset & Code Root
DATA_ROOT = None
for candidate in [
    Path("/kaggle/input/datasets/atikmorshedsohan/deepnt-plus-network-tomography-suite"),
    Path("/kaggle/input/deepnt-plus-network-tomography-suite"),
    Path("../kaggle_staging"),
    Path("kaggle_staging"),
    Path(".")
]:
    if candidate.exists() and (candidate / "deepnt_plus").exists() and (candidate / "data").exists():
        DATA_ROOT = candidate
        break

if not DATA_ROOT:
    for root, dirs, files in os.walk("/kaggle/input"):
        if "deepnt_plus" in dirs and "data" in dirs:
            DATA_ROOT = Path(root)
            break

if not DATA_ROOT:
    DATA_ROOT = Path(".")

print(f"✅ DeepNT+ Root Identified: {DATA_ROOT.resolve()}")
sys.path.insert(0, str(DATA_ROOT.resolve()))

import deepnt_plus
from deepnt_plus import data, layers
from deepnt_plus.model import DeepNTPlus, ModelConfig
from deepnt_plus.data import load_dataset, describe
print(f"✅ PyTorch Version : {torch.__version__}")
print(f"✅ CUDA Available  : {torch.cuda.is_available()}")

✅ DeepNT+ Root Identified: C:\Users\KINGDOM\Desktop\Thesis\colab\kaggle_staging
✅ PyTorch Version : 2.14.0+cpu
✅ CUDA Available  : False


## 1. Topological Datasets & GeoLite2-City Geolocation Verification
Here we inspect the 10-graph IPv4 and IPv6 datasets. Each protocol is partitioned into 10 balanced graphs (`graph_01` to `graph_10`) with a 6 Train / 2 Val / 2 Test split.
- **Strict Tomography Contract**: For every path $p$, the tomography incidence row $A_p$ must satisfy $\sum_e A_{p,e} = \text{hops}(p)$, with zero imaginary or synthetic nodes.
- **Zero-Leakage Guarantee**: Standardization statistics (mean/std of node centralities and log path counts) are strictly calculated over the training split only.
- **GeoLite2 Geolocation**: Resolved via `GeoLite2-City-CSV_20261110` with Haversine distance on edges (`use_geo=False` for pure structural track, `use_geo=True` for side-information track).

In [2]:
# Inspect datasets and print verified statistics
for proto_name, subpath in [('IPv4 (10-Graphs)', 'data/ipv4_10graphs'), ('IPv6 (10-Graphs)', 'data/ipv6_10graphs')]:
    dpath = DATA_ROOT / subpath
    if not dpath.exists():
        continue
    nf = pd.read_csv(dpath / 'node_features.csv')
    ef = pd.read_csv(dpath / 'edge_features.csv')
    pf = pd.read_csv(dpath / 'paths_all.csv')
    
    unique_n = nf['node_id'].nunique()
    u_country = nf.drop_duplicates('node_id')['country'].notna().mean() * 100
    u_coords = (nf.drop_duplicates('node_id')['latitude'].notna() & nf.drop_duplicates('node_id')['longitude'].notna()).mean() * 100
    e_geo = ef['geo_distance_km'].notna().mean() * 100 if 'geo_distance_km' in ef.columns else 0.0
    vd = ef['geo_distance_km'].dropna() if 'geo_distance_km' in ef.columns else pd.Series([])
    
    graphs_v, norm_v = load_dataset(dpath, use_geo=True)
    df_desc = describe(graphs_v)
    
    print(f"\n==================================================================")
    print(f"📊 {proto_name} Summary:")
    print(f"   Unique Nodes: {unique_n:,} | Country Res: {u_country:.1f}% | Lat/Lon Res: {u_coords:.1f}%")
    print(f"   Total Edges : {len(ef):,} | Geo Distance Edges: {e_geo:.1f}% | Median Dist: {vd.median():.1f} km")
    print(f"   Total Paths : {len(pf):,} across 10 balanced graphs (6 train / 2 val / 2 test)")
    print(f"==================================================================")
    display(df_desc)


📊 IPv4 (10-Graphs) Summary:
   Unique Nodes: 5,596 | Country Res: 100.0% | Lat/Lon Res: 99.1%
   Total Edges : 6,526 | Geo Distance Edges: 97.9% | Median Dist: 94.6 km
   Total Paths : 1,326 across 10 balanced graphs (6 train / 2 val / 2 test)
      graph  split  nodes  edges  ...  observed  hidden  outliers  median_rtt_ms
0  graph_01  train    500    525  ...        54      66         0     182.746002
1  graph_02  train    467    487  ...        54      66         1     169.300003
2  graph_03  train    552    571  ...        54      65         1     285.084991
3  graph_04  train    655    667  ...        65      79         1     125.810997
4  graph_05  train    598    607  ...        65      79         0      96.650002
5  graph_06  train    776    803  ...        63      76         0     209.324005
6  graph_07    val    605    610  ...        54      65         0     194.302002
7  graph_08    val    737    745  ...        64      79         0      47.007999
8  graph_09   test    786 

## 2. Mathematical Architecture of DeepNT+
DeepNT+ combines neural representation learning on dynamic graphs with analytical network tomography and robust Laplace likelihood uncertainty estimation:

1. **Topology Message Passing**:
   $$h_v = \text{GNN}\big(\big[x_v, \log(1 + \text{cov}_v), \bar{r}_v\big], \mathcal{E}\big)$$
   where $x_v$ comprises normalized degrees, centralities, PageRank, and k-core, while $\text{cov}_v$ and $\bar{r}_v$ inject observed probe coverage dynamically.
2. **Path Embedding**:
   $$Z_p = \text{BiLSTM}\big([h_{u_1}, h_{u_2}, \dots, h_{u_L}]\big)$$
3. **Additive Physical Prior**:
   $$\hat{y}_{\text{add}} = A_p x_e + b, \quad x_e = \text{softplus}(\text{MLP}(h_u, h_v, e_{uv})) \cdot \text{prior\_edge\_ms}$$
4. **Analytical Ridge Tomography (Convex Branch)**:
   $$\min_x \|A_{\text{ctx}} x - y_{\text{ctx}}\|^2 + \alpha \|x - x_0\|^2 \implies \hat{y}_{\text{convex}} = A_{\text{tgt}} x^*$$
5. **Dynamic Residual Gate**:
   $$\lambda_p = \sigma\big(\text{MLP}([\text{coverage}_p, \log(1 + N_{\text{ctx}}), \text{residual}])\big), \quad \hat{y} = \hat{y}_{\text{convex}} + (1 - \lambda_p)(\hat{y}_{\text{gnn}} - \hat{y}_{\text{convex}})$$
6. **Laplace Likelihood Uncertainty Calibration**:
   $$\mathcal{L} = \exp(-s) |z - \mu| + s, \quad s = \log(b)$$

In [3]:
# Demonstrate DeepNT+ forward pass on a live graph
graphs_sample, norm_sample = load_dataset(DATA_ROOT / 'data/ipv4_10graphs', use_geo=True)
g_test = graphs_sample['graph_09'] # Holdout test graph (Ghana acc-gh monitor)

cfg = ModelConfig(d=64, gnn_kind='split', tomography=True, convex=True, temporal=True, uncertainty=True)
model = DeepNTPlus(cfg, norm_sample)
model.eval()

obs_idx = torch.where(g_test.observed & g_test.ok)[0]
unobs_idx = torch.where((~g_test.observed) & g_test.ok)[0]

with torch.no_grad():
    out = model(g_test, obs_idx, unobs_idx, g_test.y[obs_idx])

df_sample = pd.DataFrame({
    'Path_ID': [g_test.path_ids[i] for i in unobs_idx[:8]],
    'Hops': g_test.hops[unobs_idx[:8]].numpy(),
    'True_RTT_ms': g_test.y[unobs_idx[:8]].numpy(),
    'Predicted_RTT_ms': out['y_hat'][:8].numpy(),
    'Neural_RTT_ms': out['y_gnn'][:8].numpy(),
    'Convex_RTT_ms': out['y_convex'][:8].numpy(),
    'Gate_Lambda': out['lam'][:8].numpy(),
    'Laplace_LogVar': out['logvar'][:8].numpy() if out['logvar'] is not None else np.nan
})

print(f"✅ Live Inference Output on Unseen Test Paths ({g_test.gid}):")
display(df_sample.round(2))

✅ Live Inference Output on Unseen Test Paths (graph_09):
           Path_ID  Hops  ...  Gate_Lambda  Laplace_LogVar
0  graph_09_p00000  18.0  ...         0.87           -0.01
1  graph_09_p00001  17.0  ...         0.87           -0.00
2  graph_09_p00005  10.0  ...         0.87           -0.03
3  graph_09_p00007  10.0  ...         0.87           -0.03
4  graph_09_p00008  12.0  ...         0.87            0.05
5  graph_09_p00010  12.0  ...         0.87            0.02
6  graph_09_p00011  11.0  ...         0.87            0.01
7  graph_09_p00015  13.0  ...         0.87            0.04

[8 rows x 8 columns]


## 3. Publication-Grade Benchmark Results (Tesla T4 GPU, 5 Seeds)
Here we display the authentic master benchmark results generated from the 5-seed evaluation on Kaggle Tesla T4 GPU (`Kernel version 10`).

In [4]:
# Load and display authentic benchmark summary CSV
res_path = DATA_ROOT / 'benchmark_results' / 'authentic_master_benchmark_summary.csv'
if not res_path.exists():
    for alt in [Path('benchmark_results/authentic_master_benchmark_summary.csv'), Path('kaggle_output_geolite2/authentic_master_benchmark_summary.csv')]:
        if alt.exists():
            res_path = alt
            break

df_results = pd.read_csv(res_path)
print("📊 100% AUTHENTIC MASTER BENCHMARK TABLE (5-Seed Mean ± Std):")
display(df_results[['Dataset', 'Model', 'Family', 'MAE', 'MAE_std', 'R2', 'Cov90', 'ECE']].round(3))

📊 100% AUTHENTIC MASTER BENCHMARK TABLE (5-Seed Mean ± Std):
                       Dataset                            Model  ...   Cov90    ECE
0   IPv4 (10-Graphs, GeoLite2)                           Median  ...     NaN    NaN
1   IPv4 (10-Graphs, GeoLite2)  Hop-count (Linear w/ Intercept)  ...     NaN    NaN
2   IPv4 (10-Graphs, GeoLite2)  Convex Tomography (Tuned Ridge)  ...     NaN    NaN
3   IPv4 (10-Graphs, GeoLite2)           GAT (Pure Spatial GNN)  ...     NaN    NaN
4   IPv4 (10-Graphs, GeoLite2)     GraphSAGE (Pure Spatial GNN)  ...     NaN    NaN
5   IPv4 (10-Graphs, GeoLite2)  DeepNT (Neural Tomo, No Convex)  ...  83.158  0.131
6   IPv4 (10-Graphs, GeoLite2)    DeepNT (Static + Convex Gate)  ...  86.842  0.049
7   IPv4 (10-Graphs, GeoLite2)     DeepNT+ (Full Dynamic, Ours)  ...  85.526  0.053
8       IPv4 (10-Graphs, Pure)                           Median  ...     NaN    NaN
9       IPv4 (10-Graphs, Pure)  Hop-count (Linear w/ Intercept)  ...     NaN    NaN
10      IPv4 (1

### Visual Comparison Charts (10-Graph IPv4 & IPv6)

In [5]:
# Display high-resolution charts
fig_dir = DATA_ROOT / 'benchmark_results'
if not fig_dir.exists():
    fig_dir = Path('kaggle_output_geolite2')

chart1 = fig_dir / 'authentic_10graphs_comparison.png'
chart2 = fig_dir / 'authentic_delay_scatter.png'
chart3 = fig_dir / 'authentic_uncertainty_calibration.png'

if chart1.exists():
    display(Markdown("### Figure 1: Test MAE Comparison (10-Graph IPv4 & IPv6 with Tuned Baselines)"))
    display(Image(filename=str(chart1), width=950))

if chart2.exists():
    display(Markdown("### Figure 2: Ground Truth vs Predicted Delay Scatter (Ideal Diagonal Line)"))
    display(Image(filename=str(chart2), width=950))

if chart3.exists():
    display(Markdown("### Figure 3: Authentic Laplace Uncertainty Calibration (Empirical vs Nominal Coverage)"))
    display(Image(filename=str(chart3), width=650))

<IPython.core.display.Markdown object>
<IPython.core.display.Image object>
<IPython.core.display.Markdown object>
<IPython.core.display.Image object>
<IPython.core.display.Markdown object>
<IPython.core.display.Image object>


## 4. Deep Dive: Pure Topological vs GeoLite2 Side-Information Track
Here we compare the effect of incorporating GeoLite2 Haversine edge distances on the identical 10-graph IPv4 benchmark:
- **`use_geo=False` (Pure Track)**: Strictly uses topological node centralities and log path counts on edges (`edge_dim=1`).
- **`use_geo=True` (GeoLite2 Track)**: Concatenates standardized Haversine edge distances (`edge_dim=2`).
Both tracks use identical 5 evaluation seeds (`[0, 1, 2, 3, 4]`) and 35 epochs.

In [6]:
# Filter and compare Pure vs GeoLite2
v4_geo = df_results[df_results['Dataset'] == 'IPv4 (10-Graphs, GeoLite2)'].set_index('Model')
v4_pure = df_results[df_results['Dataset'] == 'IPv4 (10-Graphs, Pure)'].set_index('Model')

comp_rows = []
for m in v4_pure.index:
    if m in v4_geo.index:
        p_mae = v4_pure.loc[m, 'MAE']
        p_std = v4_pure.loc[m, 'MAE_std']
        g_mae = v4_geo.loc[m, 'MAE']
        g_std = v4_geo.loc[m, 'MAE_std']
        diff = g_mae - p_mae
        comp_rows.append({
            'Model': m,
            'Pure MAE (ms)': f"{p_mae:.2f} ± {p_std:.2f}" if p_std > 0 else f"{p_mae:.2f}",
            'GeoLite2 MAE (ms)': f"{g_mae:.2f} ± {g_std:.2f}" if g_std > 0 else f"{g_mae:.2f}",
            'Diff (ms)': f"{diff:+.2f} ms",
            'Pure R2 (%)': f"{v4_pure.loc[m, 'R2']:.1f}%",
            'GeoLite2 R2 (%)': f"{v4_geo.loc[m, 'R2']:.1f}%",
            'GeoLite2 Cov90': f"{v4_geo.loc[m, 'Cov90']:.1f}%" if pd.notna(v4_geo.loc[m, 'Cov90']) else "—",
            'GeoLite2 ECE': f"{v4_geo.loc[m, 'ECE']:.3f}" if pd.notna(v4_geo.loc[m, 'ECE']) else "—",
        })

df_comp = pd.DataFrame(comp_rows)
display(HTML("<h4>IPv4 10-Graphs: Pure Topological vs GeoLite2 Side-Information</h4>"))
display(df_comp)

<IPython.core.display.HTML object>
                             Model Pure MAE (ms)  ... GeoLite2 Cov90 GeoLite2 ECE
0                           Median         61.84  ...              —            —
1  Hop-count (Linear w/ Intercept)         57.39  ...              —            —
2  Convex Tomography (Tuned Ridge)         47.64  ...              —            —
3           GAT (Pure Spatial GNN)  60.48 ± 4.44  ...              —            —
4     GraphSAGE (Pure Spatial GNN)  59.40 ± 3.90  ...              —            —
5  DeepNT (Neural Tomo, No Convex)  54.41 ± 2.56  ...          83.2%        0.131
6    DeepNT (Static + Convex Gate)  48.18 ± 1.02  ...          86.8%        0.049
7     DeepNT+ (Full Dynamic, Ours)  48.62 ± 0.67  ...          85.5%        0.053

[8 rows x 8 columns]


## 5. Source Code Inspection & Verification Suite
Reviewers can inspect the exact implementation files used in the experiments:

In [7]:
# Verify tests and display core module code
import inspect

print("=" * 80)
print("1. MODEL CONFIGURATION CONTRACT (deepnt_plus.model.ModelConfig)")
print("=" * 80)
print(inspect.getsource(ModelConfig))

print("=" * 80)
print("2. HYBRID CONVEX TOMOGRAPHY & FORWARD PASS (deepnt_plus.model.DeepNTPlus.forward)")
print("=" * 80)
lines = inspect.getsourcelines(DeepNTPlus.forward)[0]
print("".join(lines[:60]))

1. MODEL CONFIGURATION CONTRACT (deepnt_plus.model.ModelConfig)
@dataclass
class ModelConfig:
    node_dim: int = 8
    edge_dim: int = None             # 1 (path count only) or 2 (with geographical distance); None auto-detects from norm
    d: int = 64
    gnn_kind: str = 'split'          # split | sage | gat
    gnn_layers: int = 2
    heads: int = 4
    dropout: float = 0.1
    context_features: bool = True    # observation encoder: node-level summary of the context measurements
    tomography: bool = True          # additive path model y = A x + b with GNN-predicted edge latents x >= 0
    convex: bool = True              # ridge refinement of x from the context paths + learned fusion gate
    temporal: bool = False           # TGN memory + GRU + Transformer over the measurement history (causal only)
    uncertainty: bool = True         # heteroscedastic Gaussian head
    has_loss: bool = False           # packet-loss head (only trained where labels exist)
    history_k: int = 8
  

## 6. Execution Instructions for External Reviewers
To re-run training and evaluate models on Kaggle GPU:
1. Fork or clone this notebook on Kaggle.
2. Ensure GPU Accelerator (`Tesla T4`) is selected in the notebook settings.
3. The dataset `atikmorshedsohan/deepnt-plus-network-tomography-suite` is already attached.
4. To run the full 5-seed benchmark script:
```bash
python /kaggle/input/deepnt-plus-network-tomography-suite/deepnt_plus/run.py
```
Or execute the benchmark script:
```bash
python kaggle_kernel_run/run_kaggle.py
```

---
**Citation & Contact**:
Atikmorshed Sohan — *Neural Network Tomography with Dynamic Hybrid Convex Routing & Calibrated Uncertainty* (2026).